Get Mean std

In [1]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

DATA_DIR = '/mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version'
BATCH_SIZE = 64  # 批处理大小，大一点算得快
NUM_WORKERS = 4

def main():
    print(f"🚀 开始统计数据集: {DATA_DIR}")
    
    if not os.path.exists(DATA_DIR):
        raise FileNotFoundError(f"找不到目录: {DATA_DIR}")

    # 1. 简单的 Transform: 只转为 Tensor (0-1之间), 不做 Normalize
    transform = transforms.Compose([
        transforms.Resize((224, 224)), # 确保大小一致，虽然计算mean/std其实不强制resize，但为了批处理必须resize
        transforms.ToTensor()
    ])

    dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # 2. 统计类别数量
    print("-" * 30)
    print(f"总图片数: {len(dataset)}")
    print(f"类别映射: {dataset.class_to_idx}")
    print("-" * 30)
    
    # 3. 在线计算 Mean 和 Std (Welford's algorithm 或 简单的 Sum/SqSum)
    # 这里使用 Sum/SqSum 方法，对于几千张图完全够用且精确
    
    channels_sum = torch.zeros(3)
    channels_sq_sum = torch.zeros(3)
    num_pixels = 0

    print("正在遍历所有图片计算像素值...")
    
    for data, _ in tqdm(loader):
        # data shape: [Batch, 3, H, W]
        batch_size = data.size(0)
        c = data.size(1)
        h = data.size(2)
        w = data.size(3)
        
        # 累加像素总数 (N * H * W)
        num_pixels += batch_size * h * w
        
        # 累加通道和 [Batch, 3, H, W] -> [3]
        channels_sum += data.sum(dim=[0, 2, 3])
        
        # 累加通道平方和
        channels_sq_sum += (data ** 2).sum(dim=[0, 2, 3])

    # 4. 计算最终结果
    mean = channels_sum / num_pixels
    
    # Var = E[X^2] - (E[X])^2
    # Std = Sqrt(Var)
    std = (channels_sq_sum / num_pixels - mean ** 2).sqrt()

    print("\n" + "="*40)
    print("✅ 统计完成！请使用以下参数替换代码中的 transforms.Normalize")
    print("="*40)
    print(f"mean = {mean.tolist()}")
    print(f"std  = {std.tolist()}")
    print("="*40)
    
    # 格式化输出方便直接复制
    print("\n可以直接复制这行代码:")
    print(f"transforms.Normalize({[round(x, 4) for x in mean.tolist()]}, {[round(x, 4) for x in std.tolist()]})")

if __name__ == '__main__':
    main()

🚀 开始统计数据集: /mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version
------------------------------
总图片数: 4539
类别映射: {'test': 0, 'train': 1, 'val': 2}
------------------------------
正在遍历所有图片计算像素值...


 13%|█▎        | 9/71 [01:54<11:49, 11.45s/it]/mnt/disk1/anaconda/envs/Cell/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (139583899 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/mnt/disk1/anaconda/envs/Cell/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (155055316 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/mnt/disk1/anaconda/envs/Cell/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (173327462 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 14%|█▍        | 10/71 [02:53<23:35, 23.20s/it]/mnt/disk1/anaconda/envs/Cell/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (146571400 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 63%|██

KeyboardInterrupt: 

bracs train mean std

In [2]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image
import warnings

# ==========================================
# 🔥 核心修正 1: 彻底解除 PIL 的巨大图像尺寸限制，防止报错
Image.MAX_IMAGE_PIXELS = None 
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
# ==========================================

DATA_DIR = '/mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version/train' # 建议只在 train 集上算
BATCH_SIZE = 1  # 🔥 核心修正 2: 必须设为 1，因为大图尺寸不一
NUM_WORKERS = 4

def main():
    print(f"🚀 开始精准统计数据集 (原尺寸像素): {DATA_DIR}")
    
    if not os.path.exists(DATA_DIR):
        raise FileNotFoundError(f"找不到目录: {DATA_DIR}")

    # 🔥 核心修正 3: 绝对不能 Resize！保留原汁原味的病理像素分布
    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print("-" * 30)
    print(f"总图片数: {len(dataset)}")
    print(f"类别映射: {dataset.class_to_idx}")
    print("-" * 30)
    
    channels_sum = torch.zeros(3)
    channels_sq_sum = torch.zeros(3)
    num_pixels = 0

    print("正在遍历所有图片计算真实像素值 (速度可能稍慢，请耐心等待)...")
    
    for data, _ in tqdm(loader):
        # data shape: [1, 3, H, W]
        h = data.size(2)
        w = data.size(3)
        
        # 累加像素总数 (1 * H * W)
        num_pixels += h * w
        
        # 累加通道和 [1, 3, H, W] -> [3]
        channels_sum += data.sum(dim=[0, 2, 3])
        
        # 累加通道平方和
        channels_sq_sum += (data ** 2).sum(dim=[0, 2, 3])

    # 计算最终结果
    mean = channels_sum / num_pixels
    std = (channels_sq_sum / num_pixels - mean ** 2).sqrt()

    print("\n" + "="*40)
    print("✅ 统计完成！这是最原汁原味的 BRACS 数据集统计量")
    print("="*40)
    print(f"mean = {[round(x, 4) for x in mean.tolist()]}")
    print(f"std  = {[round(x, 4) for x in std.tolist()]}")
    print("="*40)

if __name__ == '__main__':
    main()

🚀 开始精准统计数据集 (原尺寸像素): /mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version/train
------------------------------
总图片数: 3657
类别映射: {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}
------------------------------
正在遍历所有图片计算真实像素值 (速度可能稍慢，请耐心等待)...


100%|██████████| 3657/3657 [06:28<00:00,  9.42it/s] 



✅ 统计完成！这是最原汁原味的 BRACS 数据集统计量
mean = [0.7265, 0.5587, 0.6954]
std  = [0.1947, 0.2282, 0.1714]


In [ ]:
import os
import shutil
from tqdm import tqdm

# --- 1. 配置路径 (请根据你的截图修改) ---
# 指向截图里的那个 breast 文件夹
# 假设你的脚本和 BreaKHis_v1 在同一级目录
SOURCE_ROOT = r"/root/autodl-tmp/lyf/BreaKHis_v1/histology_slides/breast" 
TARGET_ROOT = r"/root/autodl-tmp/lyf/BreaKHis_400X"

# 建立 文件夹名 -> 简称 的映射 (对应论文的8分类)
TYPE_MAPPING = {
    'adenosis': 'A',
    'fibroadenoma': 'F',
    'phyllodes_tumor': 'PT',
    'tubular_adenoma': 'TA',
    'ductal_carcinoma': 'DC',
    'lobular_carcinoma': 'LC',
    'mucinous_carcinoma': 'MC',
    'papillary_carcinoma': 'PC'
}

def main():
    if not os.path.exists(SOURCE_ROOT):
        print(f"错误: 找不到路径 {SOURCE_ROOT}")
        print("请检查你的 BreaKHis_v1 文件夹位置，确保路径正确。")
        return

    # 创建目标文件夹
    if not os.path.exists(TARGET_ROOT):
        os.makedirs(TARGET_ROOT)
    
    # 按照简称创建 8 个子文件夹
    for abbr in TYPE_MAPPING.values():
        os.makedirs(os.path.join(TARGET_ROOT, abbr), exist_ok=True)

    print(f"正在从 {SOURCE_ROOT} 扫描并提取 200X 图片...")
    
    count = 0
    # 遍历第1层: 肿瘤类型 (adenosis, ductal_carcinoma...)
    for type_name in os.listdir(SOURCE_ROOT):
        type_path = os.path.join(SOURCE_ROOT, type_name)
        
        # 确保是文件夹且在我们的映射表中
        if os.path.isdir(type_path) and type_name in TYPE_MAPPING:
            class_abbr = TYPE_MAPPING[type_name] # 获取简称, 如 'A'
            
            # 遍历第2层: 病人ID (SOB_B_A_14-22549AB...)
            for patient_id in os.listdir(type_path):
                patient_path = os.path.join(type_path, patient_id)
                
                if os.path.isdir(patient_path):
                    # 遍历第3层: 放大倍数 (40X, 100X...)
                    # 我们只找 '400X'
                    mag_path = os.path.join(patient_path, "400X")
                    
                    if os.path.exists(mag_path):
                        # 遍历第4层: 真正的图片
                        for img_name in os.listdir(mag_path):
                            if img_name.endswith('.png'):
                                src = os.path.join(mag_path, img_name)
                                dst = os.path.join(TARGET_ROOT, class_abbr, img_name)
                                
                                shutil.copy2(src, dst)
                                count += 1
                                if count % 100 == 0:
                                    print(f"\r已处理: {count} 张", end="")

    print(f"\n\n处理完成！共提取 {count} 张 400X 图片。")
    print(f"数据已保存在: {TARGET_ROOT}")
    print("理论上你应该有 1995 张图片 (参考论文表1)。")

if __name__ == '__main__':
    main()


In [5]:
import os
root_dir = r"E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Cleaned_Strict"
sum = 0
for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            sum += 1
print(f"Total images: {sum}")

Total images: 2047


In [6]:
import os
import shutil
from tqdm import tqdm

# --- 配置路径 ---
SOURCE_ROOT = r"E:\首都医科大学\BreakHis实验\BreaKHis_v1"        # 你的原始数据集路径
TARGET_ROOT = r"./BreaKHis_100X_Only"  # 目标路径 (只存40X)

# 8个类别简称
SUBTYPES = ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']

def main():
    if not os.path.exists(SOURCE_ROOT):
        print(f"错误: 找不到源目录 {SOURCE_ROOT}")
        return

    if not os.path.exists(TARGET_ROOT):
        os.makedirs(TARGET_ROOT)

    for subtype in SUBTYPES:
        os.makedirs(os.path.join(TARGET_ROOT, subtype), exist_ok=True)

    print("开始提取 400 倍率图片...")
    
    count = 0
    # 按照论文 Table 1 核对数量
    stats = {st: 0 for st in SUBTYPES} 

    for root, dirs, files in os.walk(SOURCE_ROOT):
        if '200X' not in root: 
            continue
            
        for file in files:
            if file.lower().endswith('.png'):
                # 匹配类别
                matched_type = None
                lower_root = root.lower()
                
                if 'adenosis' in lower_root: matched_type = 'A'
                elif 'fibroadenoma' in lower_root: matched_type = 'F'
                elif 'phyllodes_tumor' in lower_root: matched_type = 'PT'
                elif 'tubular_adenoma' in lower_root: matched_type = 'TA'
                elif 'ductal_carcinoma' in lower_root: matched_type = 'DC'
                elif 'lobular_carcinoma' in lower_root: matched_type = 'LC'
                elif 'mucinous_carcinoma' in lower_root: matched_type = 'MC'
                elif 'papillary_carcinoma' in lower_root: matched_type = 'PC'
                
                if matched_type:
                    src_path = os.path.join(root, file)
                    dst_path = os.path.join(TARGET_ROOT, matched_type, file)
                    shutil.copy2(src_path, dst_path)
                    
                    count += 1
                    stats[matched_type] += 1

    print(f"\n处理完成! 总共提取: {count} 张")
    print("各类别统计 (请核对论文表1):")
    print(stats)

if __name__ == '__main__':
    main()
    

开始提取 400 倍率图片...

处理完成! 总共提取: 2013 张
各类别统计 (请核对论文表1):
{'A': 111, 'F': 264, 'PT': 108, 'TA': 140, 'DC': 896, 'LC': 163, 'MC': 196, 'PC': 135}


In [7]:
import os
import shutil
import csv
import imagehash
from PIL import Image
from tqdm import tqdm
import glob
from collections import defaultdict

# ================= 🔧 配置区域 =================
SOURCE_DIR = r"E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Only"
CLEAN_DIR = r"E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Cleaned_Strict" 
DUPLICATE_DIR = r"E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Quarantine"
LOG_FILE = r"E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\strict_cleaning_report_100X.csv"
# ===============================================

def calculate_phash(image_path):
    try:
        with Image.open(image_path) as img:
            return str(imagehash.phash(img))
    except Exception as e:
        print(f"❌ 无法读取: {image_path}")
        return None

def copy_file_preserving_structure(src_path, root_src_dir, target_root_dir):
    relative_path = os.path.relpath(src_path, root_src_dir)
    target_path = os.path.join(target_root_dir, relative_path)
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    shutil.copy2(src_path, target_path)
    return target_path

def main():
    print("🚀 启动严格清洗模式 (Strict Mode)...")
    print("ℹ️ 逻辑: 只要发现重复，所有相关图片全部移入隔离区，不保留任何副本。")
    
    # 1. 扫描文件
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif']
    all_files = []
    print("🔍 正在扫描文件列表...")
    for ext in image_extensions:
        all_files.extend(glob.glob(os.path.join(SOURCE_DIR, '**', ext), recursive=True))
    print(f"📄 共扫描到 {len(all_files)} 张图片。")

    # 2. 计算哈希并建立索引 (Phase 1)
    print("⏳ [Phase 1/2] 正在建立哈希索引...")
    hash_map = defaultdict(list)
    
    for file_path in tqdm(all_files):
        h = calculate_phash(file_path)
        if h:
            hash_map[h].append(file_path)

    # 3. 分流复制 (Phase 2)
    print("🚚 [Phase 2/2] 正在分流文件...")
    
    count_clean = 0
    count_quarantine = 0
    
    with open(LOG_FILE, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['Status', 'Hash', 'Original_Path', 'Saved_To'])

        for h, paths in tqdm(hash_map.items()):
            # 逻辑核心：
            if len(paths) == 1:
                # === 情况 A: 独苗，绝对安全 ===
                src = paths[0]
                target = copy_file_preserving_structure(src, SOURCE_DIR, CLEAN_DIR)
                writer.writerow(['Clean', h, src, target])
                count_clean += 1
            else:
                # === 情况 B: 有重复 (哪怕只有2张)，全部隔离 ===
                # 遍历列表里的每一张图，统统扔进 Duplicate 文件夹
                for src in paths:
                    target = copy_file_preserving_structure(src, SOURCE_DIR, DUPLICATE_DIR)
                    writer.writerow(['Quarantined', h, src, target])
                    count_quarantine += 1

    print("\n" + "="*50)
    print("🎉 严格清洗完成！")
    print(f"✅ 安全数据 (Clean): {count_clean} 张 -> 存入 {CLEAN_DIR}")
    print(f"🚫 隔离数据 (Conflict): {count_quarantine} 张 -> 存入 {DUPLICATE_DIR}")
    print(f"   (包含 {len(all_files) - count_clean} 张有争议的图片)")
    print(f"📝 详细报告: {LOG_FILE}")
    print("="*50)
    print("💡 建议: 去隔离区(Quarantine)看看那些图片，检查它们的父文件夹名字，")
    print("   以此判断它们到底属于哪一类，确认后再手动移回 Clean 文件夹。")

if __name__ == '__main__':
    main()

🚀 启动严格清洗模式 (Strict Mode)...
ℹ️ 逻辑: 只要发现重复，所有相关图片全部移入隔离区，不保留任何副本。
🔍 正在扫描文件列表...
📄 共扫描到 2013 张图片。
⏳ [Phase 1/2] 正在建立哈希索引...


100%|██████████| 2013/2013 [00:11<00:00, 176.97it/s]


🚚 [Phase 2/2] 正在分流文件...


100%|██████████| 1981/1981 [00:02<00:00, 881.56it/s]


🎉 严格清洗完成！
✅ 安全数据 (Clean): 1949 张 -> 存入 E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Cleaned_Strict
🚫 隔离数据 (Conflict): 64 张 -> 存入 E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\BreaKHis_100X_Quarantine
   (包含 64 张有争议的图片)
📝 详细报告: E:\首都医科大学\BreakHis实验\lyf\BreaKHis_100X\strict_cleaning_report_100X.csv
💡 建议: 去隔离区(Quarantine)看看那些图片，检查它们的父文件夹名字，
   以此判断它们到底属于哪一类，确认后再手动移回 Clean 文件夹。


Test Bracs

In [1]:
import os
import csv
import warnings
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import timm

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    f1_score,
    balanced_accuracy_score
)

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
torch.backends.cudnn.benchmark = True


# =========================
# 1. 配置区：这里改路径
# =========================
CONFIG = {
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # 你的当前 train 配置
    "model_name": "swin_base_patch4_window12_384.ms_in22k_ft_in1k",
    "img_size": 384,
    "num_classes": 7,
    "batch_size": 32,
    "use_tta": True,

    # 数据路径（官方 ROI）
    "test_dir": r"/mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version/test",

    # 单个模型路径：改成你要测的 .pth
    "checkpoint_path": r"/mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448/epoch_36_valacc_67.95_valf1_0.6706.pth",

    # 输出目录
    "save_dir": r"/mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448"
}

MEAN = {
    'Bracs': [0.7265, 0.5587, 0.6954]
}
STD = {
    'Bracs': [0.1947, 0.2282, 0.1714]
}


# =========================
# 2. 模型定义（与你 train 对齐）
# =========================
class SpectralGatingBlock(nn.Module):
    def __init__(self, dim, h=12, w=12):
        super(SpectralGatingBlock, self).__init__()
        self.complex_weight = nn.Parameter(
            torch.randn(dim, h, w // 2 + 1, 2, dtype=torch.float32) * 0.02
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x_fft = torch.fft.rfft2(x, dim=(-2, -1), norm='ortho')
        weight = torch.view_as_complex(self.complex_weight)
        x_fft = x_fft * weight
        x_out = torch.fft.irfft2(x_fft, s=(H, W), dim=(-2, -1), norm='ortho')
        return x + x_out


class FrequencySpatialSwin(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super(FrequencySpatialSwin, self).__init__()
        self.backbone = timm.create_model(
            CONFIG['model_name'],
            pretrained=pretrained,
            num_classes=0
        )
        self.num_features = self.backbone.num_features

        for param in self.backbone.parameters():
            param.requires_grad = False
        for name, param in self.backbone.named_parameters():
            if "layers.2" in name or "layers.3" in name or "norm" in name:
                param.requires_grad = True

        self.freq_block = SpectralGatingBlock(self.num_features, h=12, w=12)

        self.fusion_gate = nn.Sequential(
            nn.Linear(self.num_features * 2, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.GELU()
        )

        self.head = nn.Linear(self.num_features, num_classes)

    def forward(self, x):
        feat_raw = self.backbone.forward_features(x)

        if feat_raw.ndim == 4:
            B, H, W, C = feat_raw.shape
            feat_spatial = feat_raw.permute(0, 3, 1, 2)
        elif feat_raw.ndim == 3:
            B, L, C = feat_raw.shape
            H = W = int(L ** 0.5)
            feat_spatial = feat_raw.transpose(1, 2).reshape(B, C, H, W)
        else:
            raise ValueError(f"Unexpected feature shape: {feat_raw.shape}")

        spatial_vec = feat_spatial.mean(dim=[2, 3])
        freq_feat = self.freq_block(feat_spatial)
        freq_vec = freq_feat.mean(dim=[2, 3])

        concat = torch.cat([spatial_vec, freq_vec], dim=1)
        fused_vec = self.fusion_gate(concat)
        out = self.head(fused_vec)
        return out


# =========================
# 3. 预处理（与你 train 的 val/test 对齐）
# =========================
class ResizeKeepRatioPad:
    def __init__(self, size, fill=(255, 255, 255)):
        self.size = size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        new_w = int(round(w * scale))
        new_h = int(round(h * scale))

        img = TF.resize(img, (new_h, new_w), antialias=True)

        pad_w = self.size - new_w
        pad_h = self.size - new_h

        left = pad_w // 2
        right = pad_w - left
        top = pad_h // 2
        bottom = pad_h - top

        img = TF.pad(img, [left, top, right, bottom], fill=self.fill)
        return img


def build_test_loader():
    test_tf = transforms.Compose([
        ResizeKeepRatioPad(CONFIG["img_size"]),
        transforms.ToTensor(),
        transforms.Normalize(MEAN['Bracs'], STD['Bracs'])
    ])

    test_ds = datasets.ImageFolder(CONFIG["test_dir"], transform=test_tf)

    test_dl = DataLoader(
        test_ds,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4
    )

    return test_ds, test_dl


# =========================
# 4. TTA 推理
# =========================
def get_probs_with_tta(model, loader, device, use_tta=True):
    model.eval()
    all_probs = []
    all_labels = []

    use_amp = device.type == "cuda"

    with torch.inference_mode():
        for imgs, lbls in loader:
            imgs = imgs.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                if use_tta:
                    out1 = model(imgs)
                    out2 = model(torch.flip(imgs, dims=[3]))  # 水平翻转
                    out3 = model(torch.flip(imgs, dims=[2]))  # 垂直翻转
                    out_avg = (out1 + out2 + out3) / 3.0
                else:
                    out_avg = model(imgs)

                probs = F.softmax(out_avg, dim=1)

            all_probs.append(probs.cpu())
            all_labels.extend(lbls.cpu().numpy())

    return torch.cat(all_probs), np.array(all_labels)


# =========================
# 5. 评估函数
# =========================
def evaluate_test_metrics(probs, labels, class_names):
    preds = probs.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(labels, preds) * 100
    macro_f1 = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    bal_acc = balanced_accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    kappa = cohen_kappa_score(labels, preds)

    report_dict = classification_report(
        labels,
        preds,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(
        labels,
        preds,
        labels=list(range(len(class_names)))
    )

    return {
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "bal_acc": bal_acc,
        "mcc": mcc,
        "kappa": kappa,
        "report_dict": report_dict,
        "cm": cm,
        "preds": preds
    }


# =========================
# 6. 保存结果
# =========================
SUMMARY_FIELDS = [
    "checkpoint",
    "Acc",
    "Macro_F1",
    "Weighted_F1",
    "Balanced_Acc",
    "MCC",
    "Kappa",
    "TTA"
]


def save_summary_csv(save_dir, checkpoint_name, metrics, use_tta):
    csv_path = os.path.join(save_dir, "test_summary.csv")
    row_data = {
        "checkpoint": checkpoint_name,
        "Acc": f"{metrics['acc']:.2f}%",
        "Macro_F1": f"{metrics['macro_f1']:.4f}",
        "Weighted_F1": f"{metrics['weighted_f1']:.4f}",
        "Balanced_Acc": f"{metrics['bal_acc']:.4f}",
        "MCC": f"{metrics['mcc']:.4f}",
        "Kappa": f"{metrics['kappa']:.4f}",
        "TTA": use_tta
    }

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_data)


# =========================
# 7. 主流程
# =========================
def main():
    os.makedirs(CONFIG["save_dir"], exist_ok=True)

    print("🚀 开始测试...")
    print(f"📦 Checkpoint: {CONFIG['checkpoint_path']}")
    print(f"🧪 Test dir   : {CONFIG['test_dir']}")

    test_ds, test_dl = build_test_loader()
    class_names = test_ds.classes

    print(f"📊 类别映射: {test_ds.class_to_idx}")

    model = FrequencySpatialSwin(
        num_classes=CONFIG["num_classes"],
        pretrained=False
    ).to(CONFIG["device"])

    ckpt = torch.load(CONFIG["checkpoint_path"], map_location=CONFIG["device"])
    model.load_state_dict(ckpt, strict=True)

    probs, labels = get_probs_with_tta(
        model,
        test_dl,
        CONFIG["device"],
        use_tta=CONFIG["use_tta"]
    )

    metrics = evaluate_test_metrics(probs, labels, class_names)

    print("\n================ 测试结果 ================")
    print(
        f"👉 Acc: {metrics['acc']:.2f}% | "
        f"Macro-F1: {metrics['macro_f1']:.4f} | "
        f"Weighted-F1: {metrics['weighted_f1']:.4f} | "
        f"Bal-Acc: {metrics['bal_acc']:.4f} | "
        f"MCC: {metrics['mcc']:.4f} | "
        f"Kappa: {metrics['kappa']:.4f}"
    )

    checkpoint_name = os.path.basename(CONFIG["checkpoint_path"]).replace(".pth", "")

    # 保存 classification report
    report_df = pd.DataFrame(metrics["report_dict"]).T
    report_save_path = os.path.join(
        CONFIG["save_dir"],
        f"{checkpoint_name}_classification_report.csv"
    )
    report_df.to_csv(report_save_path, encoding="utf-8-sig")

    # 保存 confusion matrix
    cm_df = pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)
    cm_save_path = os.path.join(
        CONFIG["save_dir"],
        f"{checkpoint_name}_confusion_matrix.csv"
    )
    cm_df.to_csv(cm_save_path, encoding="utf-8-sig")

    # 保存 summary
    save_summary_csv(
        CONFIG["save_dir"],
        checkpoint_name,
        metrics,
        CONFIG["use_tta"]
    )

    print(f"\n✅ classification report 已保存: {report_save_path}")
    print(f"✅ confusion matrix 已保存    : {cm_save_path}")
    print(f"✅ summary 已保存             : {os.path.join(CONFIG['save_dir'], 'test_summary.csv')}")


if __name__ == "__main__":
    main()

/mnt/disk1/anaconda/envs/Cell/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 开始测试...
📦 Checkpoint: /mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448/epoch_36_valacc_67.95_valf1_0.6706.pth
🧪 Test dir   : /mnt/disk1/lyf/BRACS/histoimage.na.icar.cnr.it/BRACS_RoI/latest_version/test
📊 类别映射: {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}

================ 测试结果 ================
👉 Acc: 61.40% | Macro-F1: 0.6094 | Weighted-F1: 0.6105 | Bal-Acc: 0.6130 | MCC: 0.5504 | Kappa: 0.5497

✅ classification report 已保存: /mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448/epoch_36_valacc_67.95_valf1_0.6706_classification_report.csv
✅ confusion matrix 已保存    : /mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448/epoch_36_valacc_67.95_valf1_0.6706_confusion_matrix.csv
✅ summary 已保存             : /mnt/disk1/lyf/BreakHis/Results/BRACS/20260304_235448/test_summary.csv
